# Final Comparison: DFL under Partial Feedback — Shortest Path

**Baselines (no DFL):**
- `Random` — Random feasible solution each round (no learning)
- `Greedy` — Bayesian regression, greedy exploitation (beta=0)
- `E-Greedy` — Epsilon-greedy, MSE loss, masked gradient
- `LinUCB` — Pure Bayesian linear UCB (no neural features, no DFL)

**DFL Agents (SGD-trained, epsilon-greedy):**
- `FullMSE` — 2-layer NN, epsilon-greedy, MSE loss
- `FullSPO` — 2-layer NN, epsilon-greedy, SPO+ loss

**DFL Agents (Bayesian decoupled):**
- `BayesianMSE` — Bayesian decoupled, MSE loss
- `BayesianMSE-TSMask` — Bayesian decoupled, MSE loss, Thompson sampling imputation + mask
- `BayesianSPO` — Bayesian decoupled, SPO+ loss, masked gradient
- `BayesianSPO-TSMask` — Bayesian decoupled, SPO+ loss, Thompson sampling imputation + mask
- `BayesianSPO-DI` — Bayesian decoupled, SPO+, discriminative imputation

**Dataset:** Shortest Path (8x8 grid = 112 edges, minimization, partial feedback)

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import jax
import jax.numpy as jnp
import optax
import pandas as pd
from lets_plot import *

LetsPlot.setup_html()

from src.envs.shortest_path import ShortestPathEdgeEnv
from src.agents.base import run_experiment
from src.solver import solve_shortest_path_edges, ShortestPathSolver
from src.agents.bayesian_utils import (
    init_per_arm_regression,
    compute_ucb_values,
    compute_bayesian_predictions,
    update_bayesian_stats,
)

## Loss Gradient Functions

In [2]:
def spo_grad(c_pred, c_true, solver, is_maximize=True):
    """SPO+ gradient w.r.t. predictions."""
    cp = np.asarray(c_pred, dtype=np.float64)
    ct = np.asarray(c_true, dtype=np.float64)
    n = cp.shape[0]
    w_true = solver(ct)
    w_spo = solver(2 * cp - ct)
    if is_maximize:
        return jnp.array(2.0 * (w_spo - w_true) / n, dtype=jnp.float32)
    else:
        return jnp.array(2.0 * (w_true - w_spo) / n, dtype=jnp.float32)


def mse_grad(c_pred, c_true):
    """MSE gradient w.r.t. predictions."""
    n = c_pred.shape[0]
    return 2.0 * (c_pred - c_true) / n


def make_spo_grad(solver, is_maximize=True):
    return lambda c_pred, c_true: spo_grad(c_pred, c_true, solver, is_maximize)


def make_poly_decay_schedule(init_lr, power=0.25):
    def schedule(count):
        return init_lr / jnp.power(count + 1, power)
    return schedule

## JIT Kernels

In [3]:
@jax.jit
def _fused_ucb(X, W1, b1, A_inv, b_bayes, beta):
    h = X @ W1 + b1
    theta = jnp.einsum('nij,nj->ni', A_inv, b_bayes)
    means = theta @ h
    Ax = jnp.einsum('nij,j->ni', A_inv, h)
    stds = jnp.sqrt(jnp.maximum(jnp.einsum('i,ni->n', h, Ax), 0.0))
    return means + beta * stds

@jax.jit
def _fused_predict(X, W1, b1, A_inv, b_bayes):
    h = X @ W1 + b1
    theta = jnp.einsum('nij,nj->ni', A_inv, b_bayes)
    return theta @ h

@jax.jit
def _fused_bayesian_core(X, W1, b1, A, b_bayes, A_inv, rewards, mask):
    h = X @ W1 + b1
    A_new, b_new, A_inv_new = update_bayesian_stats(
        A, b_bayes, A_inv, h, rewards, mask)
    theta = jnp.einsum('nij,nj->ni', A_inv_new, b_new)
    pred = theta @ h
    return h, pred, theta, A_new, b_new, A_inv_new

@jax.jit
def _bayesian_grads(X, theta, grad_pred):
    grad_h = jnp.einsum('ni,n->i', theta, grad_pred)
    return {'W1': jnp.outer(X, grad_h), 'b1': grad_h}

def _make_step_fn(optimizer):
    @jax.jit
    def step(grads, opt_state, params):
        updates, new_state = optimizer.update(grads, opt_state, params)
        return optax.apply_updates(params, updates), new_state
    return step

## Agents

In [4]:
class LinearBayesianAgent:
    """1-hidden-layer features + per-arm Bayesian regression + UCB exploration.

    Supports imputation modes:
      'model'        -- posterior mean on unobserved
      'thompson'     -- Thompson sampling: average gradient over posterior samples, then mask
      'disc_explore'  -- DX: masked gradient (no special exploration)
      'disc_impute'   -- DI: adversarial imputation on discriminating edges (gradient only)

    decision_focused_ucb: when True, weight UCB bonus by φ(slack/σ) to suppress
      exploration on non-contested arms (DFE).

    DI only modifies the gradient in update() — it uses standard UCB for act().
    When both DFE and DI are active:
      - DFE controls act() (slack-weighted exploration)
      - DI controls update() (adversarial imputation gradient)
    """

    def __init__(self, context_dim, n_arms, weights, capacity,
                 lr, loss_grad_fn, hidden_dim=32, beta=1.0, lambda_reg=1.0,
                 weight_decay=1e-4, max_grad_norm=5.0, imputation='model',
                 mask_grad=False, minimize=False, solver=None, seed=None,
                 lr_schedule=None, co_solver=None,
                 contested_alpha=1.0, contested_eps=1e-6,
                 disc_K=5, disc_lambda=0.3,
                 n_thompson_samples=20,
                 decision_focused_ucb=False, **kwargs):
        self.n_arms = n_arms
        self.weights_np = np.asarray(weights, dtype=np.float64)
        self.capacity = capacity
        self.beta = beta
        self.loss_grad_fn = loss_grad_fn
        self.imputation = imputation
        self.mask_grad = mask_grad
        self.minimize = minimize
        self.t = 0
        self.n_thompson_samples = n_thompson_samples
        self.decision_focused_ucb = decision_focused_ucb

        self.co_solver = co_solver
        self.contested_alpha = contested_alpha
        self.contested_eps = contested_eps
        self.disc_K = disc_K
        self.disc_lambda = disc_lambda
        self.contested_history = []

        if imputation in ('disc_explore', 'disc_impute') and co_solver is None:
            raise ValueError('co_solver required for disc_explore/disc_impute')
        if decision_focused_ucb and co_solver is None:
            raise ValueError('co_solver required for decision_focused_ucb')

        if solver is None:
            raise ValueError('solver must be provided for shortest path')
        self.solver = solver

        key = jax.random.PRNGKey(seed if seed is not None else 0)
        key, init_key = jax.random.split(key)
        self.rng_key = key
        self.W1 = jax.random.normal(init_key, (context_dim, hidden_dim)) * 0.1
        self.b1 = jnp.zeros(hidden_dim)

        lr_or_schedule = lr_schedule if lr_schedule is not None else lr
        self.optimizer = optax.chain(
            optax.clip_by_global_norm(max_grad_norm),
            optax.adamw(lr_or_schedule, weight_decay=weight_decay),
        )
        self.opt_state = self.optimizer.init({'W1': self.W1, 'b1': self.b1})
        self._step = _make_step_fn(self.optimizer)

        self.A, self.b_bayes, self.A_inv = init_per_arm_regression(
            n_arms, hidden_dim, lambda_reg)

    def predict_mean(self, X):
        X_j = jnp.asarray(X, dtype=jnp.float32)
        return np.asarray(_fused_predict(X_j, self.W1, self.b1,
                                         self.A_inv, self.b_bayes))

    def act(self, X):
        self.t += 1
        X_j = jnp.asarray(X, dtype=jnp.float32)

        # --- DFE branch: slack-weighted exploration ---
        if self.decision_focused_ucb:
            h = X_j @ self.W1 + self.b1
            theta = jnp.einsum('nij,nj->ni', self.A_inv, self.b_bayes)
            means = theta @ h
            Ax = jnp.einsum('nij,j->ni', self.A_inv, h)
            stds = jnp.sqrt(jnp.maximum(jnp.einsum('i,ni->n', h, Ax), 0.0))

            means_np = np.asarray(means, dtype=np.float64)
            stds_np = np.asarray(stds, dtype=np.float64)

            slacks = self.co_solver.compute_slacks(means_np)
            on_solution = slacks < 1e-8

            phi = np.exp(-slacks / (stds_np + self.contested_eps))
            phi[on_solution] = 0.0

            phi_j = jnp.array(phi, dtype=jnp.float32)
            values = means + self.beta * stds * phi_j
            return self.solver(np.asarray(values))

        # --- Standard UCB (used by all non-DFE agents, including DI) ---
        values = _fused_ucb(X_j, self.W1, self.b1,
                            self.A_inv, self.b_bayes, self.beta)
        return self.solver(np.asarray(values))

    def update(self, X, selection, rewards, lr=None, full_feedback=False):
        X_j = jnp.asarray(X, dtype=jnp.float32)
        sel_j = jnp.asarray(selection, dtype=jnp.float32)
        rew_j = jnp.asarray(rewards, dtype=jnp.float32)
        sel_mask = sel_j > 0.5
        obs_mask = jnp.ones(self.n_arms, dtype=jnp.bool_) if full_feedback else sel_mask

        h, pred, theta, self.A, self.b_bayes, self.A_inv = \
            _fused_bayesian_core(X_j, self.W1, self.b1,
                self.A, self.b_bayes, self.A_inv, rew_j, obs_mask)

        if self.imputation == 'disc_explore':
            grad_pred = self._grad_disc_explore(pred, rew_j, obs_mask)
        elif self.imputation == 'disc_impute':
            Ax = jnp.einsum('nij,j->ni', self.A_inv, h)
            stds = jnp.sqrt(jnp.maximum(jnp.einsum('i,ni->n', h, Ax), 0.0))
            grad_pred = self._grad_disc_impute(pred, rew_j, obs_mask, stds)
        elif self.imputation == 'thompson':
            Ax = jnp.einsum('nij,j->ni', self.A_inv, h)
            stds = jnp.sqrt(jnp.maximum(jnp.einsum('i,ni->n', h, Ax), 0.0))
            grad_pred = self._grad_thompson(pred, rew_j, obs_mask, stds)
        else:
            Y_target = jnp.where(obs_mask, rew_j, pred)
            grad_pred = self.loss_grad_fn(pred, Y_target)
            if self.mask_grad:
                grad_pred = jnp.where(obs_mask, grad_pred, 0.0)

        grads = _bayesian_grads(X_j, theta, grad_pred)
        params = {'W1': self.W1, 'b1': self.b1}
        new_params, self.opt_state = self._step(grads, self.opt_state, params)
        self.W1 = jnp.clip(new_params['W1'], -10, 10)
        self.b1 = new_params['b1']

    def _grad_thompson(self, pred, rew_j, obs_mask, stds):
        """Thompson sampling imputation: draw posterior samples, average gradients, mask."""
        grad_pred = jnp.zeros(self.n_arms)
        for _ in range(self.n_thompson_samples):
            self.rng_key, subkey = jax.random.split(self.rng_key)
            z = jax.random.normal(subkey, shape=(self.n_arms,))
            imputed = pred + stds * z
            Y_target = jnp.where(obs_mask, rew_j, imputed)
            grad_pred = grad_pred + self.loss_grad_fn(pred, Y_target)
        grad_pred = grad_pred / self.n_thompson_samples
        if self.mask_grad:
            grad_pred = jnp.where(obs_mask, grad_pred, 0.0)
        return grad_pred

    def _grad_disc_explore(self, pred, rew_j, obs_mask):
        Y_target = jnp.where(obs_mask, rew_j, pred)
        grad_pred = self.loss_grad_fn(pred, Y_target)
        return jnp.where(obs_mask, grad_pred, 0.0)

    def _grad_disc_impute(self, pred, rew_j, obs_mask, stds):
        pred_np = np.asarray(pred, dtype=np.float64)
        stds_np = np.asarray(stds, dtype=np.float64)

        opt_sol, alts = self.co_solver.extract_k_alternatives(
            pred_np, self.disc_K)

        if not alts:
            return self._grad_disc_explore(pred, rew_j, obs_mask)

        grad_pred = jnp.zeros(self.n_arms)
        all_disc = np.zeros(self.n_arms, dtype=bool)

        for alt_sol, _, sym_diff in alts:
            imputed = pred_np.copy()
            opt_only = (opt_sol > 0.5) & (alt_sol < 0.5)
            alt_only = (alt_sol > 0.5) & (opt_sol < 0.5)

            imputed[opt_only] += self.contested_alpha * stds_np[opt_only]
            imputed[alt_only] -= self.contested_alpha * stds_np[alt_only]

            Y_k = np.where(np.asarray(obs_mask), np.asarray(rew_j), imputed)
            grad_k = self.loss_grad_fn(pred, jnp.array(Y_k, dtype=jnp.float32))

            disc_k = jnp.array(sym_diff)
            grad_k = jnp.where(
                obs_mask, grad_k,
                jnp.where(disc_k, self.disc_lambda * grad_k, 0.0))

            grad_pred = grad_pred + grad_k
            all_disc |= sym_diff

        self.contested_history.append(int(all_disc.sum()))
        return grad_pred / len(alts)

In [5]:
class RandomAgent:
    def __init__(self, n_arms, solver, seed=None, **kwargs):
        self.n_arms = n_arms
        self.solver = solver
        self.rng = np.random.default_rng(seed if seed is not None else 0)

    def predict_mean(self, X):
        return np.zeros(self.n_arms)

    def act(self, X):
        return self.solver(self.rng.standard_normal(self.n_arms))

    def update(self, X, selection, rewards, lr=None, full_feedback=False):
        pass


class LinearEpsilonGreedyAgent:
    """2-layer NN with decaying epsilon-greedy exploration. No Bayesian regression."""

    def __init__(self, context_dim, n_arms, weights, capacity,
                 loss_grad_fn, hidden_dim=32, epsilon0=1.0, lr=0.01,
                 weight_decay=1e-4, max_grad_norm=5.0, mask_grad=False,
                 minimize=False, solver=None, seed=None, lr_schedule=None, **kwargs):
        self.n_arms = n_arms
        self.epsilon0 = epsilon0
        self.loss_grad_fn = loss_grad_fn
        self.mask_grad = mask_grad
        self.minimize = minimize
        self.t = 0
        self.solver = solver

        key = jax.random.PRNGKey(seed if seed is not None else 0)
        key, k1 = jax.random.split(key)
        self.W1 = jax.random.normal(k1, (context_dim, hidden_dim)) * 0.01
        self.b1 = jnp.zeros(hidden_dim)
        key, k2 = jax.random.split(key)
        self.W2 = jax.random.normal(k2, (hidden_dim, n_arms)) * 0.01
        self.b2 = jnp.zeros(n_arms)
        self.rng = np.random.default_rng(seed if seed is not None else 0)

        lr_or_schedule = lr_schedule if lr_schedule is not None else lr
        self.optimizer = optax.chain(
            optax.clip_by_global_norm(max_grad_norm),
            optax.adam(lr_or_schedule),
        )
        params = {'W1': self.W1, 'b1': self.b1, 'W2': self.W2, 'b2': self.b2}
        self.opt_state = self.optimizer.init(params)

    def _predict(self, X):
        h = X @ self.W1 + self.b1
        return h @ self.W2 + self.b2, h

    def predict_mean(self, X):
        pred, _ = self._predict(jnp.asarray(X, dtype=jnp.float32))
        return np.asarray(pred)

    def act(self, X):
        self.t += 1
        pred = self.predict_mean(X)
        eps = self.epsilon0 / np.sqrt(self.t)
        if self.rng.random() < eps:
            return self.solver(pred + self.rng.standard_normal(self.n_arms))
        return self.solver(pred)

    def update(self, X, selection, rewards, lr=None, full_feedback=False):
        X_j = jnp.asarray(X, dtype=jnp.float32)
        sel_j = jnp.asarray(selection, dtype=jnp.float32)
        rew_j = jnp.asarray(rewards, dtype=jnp.float32)
        obs_mask = jnp.ones(self.n_arms, dtype=jnp.bool_) if full_feedback else (sel_j > 0.5)

        pred, h = self._predict(X_j)
        Y_target = jnp.where(obs_mask, rew_j, pred)
        grad_pred = self.loss_grad_fn(pred, Y_target)
        if self.mask_grad:
            grad_pred = jnp.where(obs_mask, grad_pred, 0.0)

        grad_W2 = jnp.outer(h, grad_pred)
        grad_b2 = grad_pred
        grad_h = self.W2 @ grad_pred
        grads = {'W1': jnp.outer(X_j, grad_h), 'b1': grad_h,
                 'W2': grad_W2, 'b2': grad_b2}
        params = {'W1': self.W1, 'b1': self.b1, 'W2': self.W2, 'b2': self.b2}

        updates, self.opt_state = self.optimizer.update(grads, self.opt_state, params)
        new_params = optax.apply_updates(params, updates)
        self.W1 = jnp.clip(new_params['W1'], -10, 10)
        self.b1 = new_params['b1']
        self.W2 = jnp.clip(new_params['W2'], -10, 10)
        self.b2 = new_params['b2']


class LinUCB:
    """Bayesian linear UCB with optional random feature projection."""

    def __init__(self, context_dim, n_arms, weights, capacity,
                 beta=1.0, lambda_reg=1.0, solver=None, minimize=False,
                 hidden_dim=None, seed=None, **kwargs):
        self.n_arms = n_arms
        self.beta = beta
        self.minimize = minimize
        self.solver = solver

        if hidden_dim is not None:
            key = jax.random.PRNGKey(seed if seed is not None else 0)
            _, init_key = jax.random.split(key)
            self.W_proj = jax.random.normal(init_key, (context_dim, hidden_dim)) * 0.1
            self.b_proj = jnp.zeros(hidden_dim)
            feat_dim = hidden_dim
        else:
            self.W_proj = None
            self.b_proj = None
            feat_dim = context_dim

        self.A, self.b_bayes, self.A_inv = init_per_arm_regression(
            n_arms, feat_dim, lambda_reg)

    def _features(self, X):
        h = jnp.asarray(X, dtype=jnp.float32)
        if self.W_proj is not None:
            h = h @ self.W_proj + self.b_proj
        return h

    def predict_mean(self, X):
        h = self._features(X)
        return np.asarray(compute_bayesian_predictions(h, self.A_inv, self.b_bayes))

    def act(self, X):
        h = self._features(X)
        ucb = compute_ucb_values(h, self.A_inv, self.b_bayes, self.beta)
        return self.solver(np.asarray(ucb))

    def update(self, X, selection, rewards, lr=None, full_feedback=False):
        h = self._features(X)
        rew_j = jnp.asarray(rewards, dtype=jnp.float32)
        mask = jnp.ones(self.n_arms, dtype=jnp.bool_) if full_feedback else (
            jnp.asarray(selection, dtype=jnp.float32) > 0.5)
        self.A, self.b_bayes, self.A_inv = update_bayesian_stats(
            self.A, self.b_bayes, self.A_inv, h, rew_j, mask)

## Experiment Setup

In [6]:
GRID_SIZE = 8
NOISE_LEVELS = [0.25]

DATASETS = {}
for nw in NOISE_LEVELS:
    nw_tag = str(nw).replace('.', '')
    DATASETS[f'sp_nw{nw_tag}'] = dict(
        env_cls=ShortestPathEdgeEnv,
        env_kwargs=dict(grid_size=GRID_SIZE, context_dim=5, noise_width=nw),
        minimize=True,
        group='shortest_path',
        noise=nw,
    )

HP = dict(lr=0.001, hidden_dim=32, lambda_reg=1.0, beta=1.0, epsilon0=1.0)

N_ROUNDS = 2000
# SEEDS = [0, 1, 2, 5, 6]
SEEDS = [0, 1, 5]

EVAL_INTERVAL = 100

## Run Experiments

In [7]:
def make_agents(env, seed, minimize):
    solver = env.solve
    # For shortest-path (minimization): negative beta → LCB
    bayes_beta = -HP['beta'] if minimize else HP['beta']
    spo = make_spo_grad(solver, is_maximize=not minimize)
    sched = make_poly_decay_schedule(HP['lr'])
    co_solver = ShortestPathSolver(GRID_SIZE)

    common = dict(context_dim=env.context_dim, n_arms=env.n_arms,
                  weights=env.weights, capacity=env.capacity)
    bayes = dict(hidden_dim=HP['hidden_dim'], lr=HP['lr'], seed=seed,
                 beta=bayes_beta, lambda_reg=HP['lambda_reg'], solver=solver,
                 minimize=minimize)
    egreedy = dict(hidden_dim=HP['hidden_dim'], lr=HP['lr'], seed=seed,
                   epsilon0=HP['epsilon0'], solver=solver, minimize=minimize)

    return {
        # --- Baselines (no DFL) ---
        'Greedy':      LinUCB(**common, beta=0.0, lambda_reg=HP['lambda_reg'],
                              solver=solver, minimize=minimize,
                              hidden_dim=HP['hidden_dim'], seed=seed),

        # --- DFL with epsilon-greedy (2-layer NN, no Bayesian decoupling) ---
        'FullMSE':     LinearEpsilonGreedyAgent(**common, **egreedy,
                                                loss_grad_fn=mse_grad,
                                                lr_schedule=sched),
        'FullSPO':     LinearEpsilonGreedyAgent(**common, **egreedy,
                                                loss_grad_fn=spo,
                                                mask_grad=True,
                                                lr_schedule=sched),

        # --- DFL with Bayesian decoupling ---
        'BayesianMSE': LinearBayesianAgent(**common, **bayes, loss_grad_fn=mse_grad,
                                           imputation='model', lr_schedule=sched),
        'BayesianSPO': LinearBayesianAgent(**common, **bayes, loss_grad_fn=spo,
                                           imputation='model', mask_grad=True,
                                           lr_schedule=sched),
        'BayesianSPO-DI': LinearBayesianAgent(**common, **bayes, loss_grad_fn=spo,
                                              imputation='disc_impute',
                                              co_solver=co_solver,
                                              contested_alpha=0.5, disc_K=5,
                                              disc_lambda=0.1, lr_schedule=sched),

        # --- Decision-Focused Exploration (DFE) ---
        'BayesianSPO-DFE': LinearBayesianAgent(**common, **bayes, loss_grad_fn=spo,
                                               imputation='model', mask_grad=True,
                                               co_solver=co_solver,
                                               decision_focused_ucb=True,
                                               lr_schedule=sched),
        'BayesianSPO-DI+DFE': LinearBayesianAgent(**common, **bayes, loss_grad_fn=spo,
                                                   imputation='disc_impute',
                                                   co_solver=co_solver,
                                                   contested_alpha=0.5, disc_K=5,
                                                   disc_lambda=0.1,
                                                   decision_focused_ucb=True,
                                                   lr_schedule=sched),
        # --- Decision-Focused Exploration (DFE) ---
        # 'BayesianMSE-DI': LinearBayesianAgent(**common, **bayes, loss_grad_fn=mse_grad,
        #                                       imputation='disc_impute',
        #                                       co_solver=co_solver,
        #                                       contested_alpha=0.5, disc_K=5,
        #                                       disc_lambda=0.1, lr_schedule=sched),
        # 'BayesianMSE-DFE': LinearBayesianAgent(**common, **bayes, loss_grad_fn=mse_grad,
        #                                        imputation='model', mask_grad=True,
        #                                        co_solver=co_solver,
        #                                        decision_focused_ucb=True,
        #                                        lr_schedule=sched),
        'BayesianMSE-DI+DFE': LinearBayesianAgent(**common, **bayes, loss_grad_fn=mse_grad,
                                                   imputation='disc_impute',
                                                   co_solver=co_solver,
                                                   contested_alpha=0.5, disc_K=5,
                                                   disc_lambda=0.1,
                                                   decision_focused_ucb=True,
                                                   lr_schedule=sched),
    }


all_agents = {}
all_results = {}

for ds_name, ds_cfg in DATASETS.items():
    env_cls = ds_cfg['env_cls']
    env_kwargs = ds_cfg['env_kwargs']
    minimize = ds_cfg['minimize']

    print(f'\n=== {ds_name} (minimize={minimize}) ===')
    all_results[ds_name] = {}
    all_agents[ds_name] = {}

    for seed in SEEDS:
        agents = make_agents(env_cls(**env_kwargs, seed=seed), seed, minimize=minimize)

        for agent_name, agent in agents.items():
            if agent_name not in all_results[ds_name]:
                all_results[ds_name][agent_name] = {}
                all_agents[ds_name][agent_name] = {}

            env_run = env_cls(**env_kwargs, seed=seed)
            result = run_experiment(
                env_run, agent, n_rounds=N_ROUNDS,
                lr=HP['lr'], full_feedback=False,
                eval_interval=EVAL_INTERVAL)
            all_results[ds_name][agent_name][seed] = result
            all_agents[ds_name][agent_name][seed] = agent
            print(f'  {agent_name:<20s} seed={seed}  final_regret={result["final_regret"]:.1f}')


=== sp_nw025 (minimize=True) ===
  Greedy               seed=0  final_regret=26881.7
  FullMSE              seed=0  final_regret=19074.8
  FullSPO              seed=0  final_regret=18881.5
  BayesianMSE          seed=0  final_regret=4342.4
  BayesianSPO          seed=0  final_regret=4020.9
  BayesianSPO-DI       seed=0  final_regret=4054.3
  BayesianSPO-DFE      seed=0  final_regret=3829.8
  BayesianSPO-DI+DFE   seed=0  final_regret=3858.3
  BayesianMSE-DI+DFE   seed=0  final_regret=4006.7
  Greedy               seed=1  final_regret=22820.9
  FullMSE              seed=1  final_regret=17203.2
  FullSPO              seed=1  final_regret=16974.5
  BayesianMSE          seed=1  final_regret=4033.9
  BayesianSPO          seed=1  final_regret=3719.3
  BayesianSPO-DI       seed=1  final_regret=3654.8
  BayesianSPO-DFE      seed=1  final_regret=3338.8
  BayesianSPO-DI+DFE   seed=1  final_regret=3483.2
  BayesianMSE-DI+DFE   seed=1  final_regret=3663.3
  Greedy               seed=5  final_regre

## Plot Results

In [8]:
from lets_plot import gggrid, ggsave

FIGDIR = 'lets-plot-images'
os.makedirs(FIGDIR, exist_ok=True)

AGENT_ORDER = [
    'Random', 'Greedy', 'E-Greedy', 'LinUCB',
    'FullMSE', 'FullSPO',
    'BayesianMSE', 'BayesianMSE-TSMask',
    'BayesianSPO', 'BayesianSPO-TSMask', 'BayesianSPO-DI',
    'BayesianSPO-DFE', 'BayesianSPO-DI+DFE', 'BayesianMSE-DI',
    'BayesianMSE-DFE', 'BayesianMSE-DI+DFE',
]

COLORS = {
    'Random':             '#7f7f7f',
    'Greedy':             '#c7c7c7',
    'E-Greedy':           '#bcbd22',
    'LinUCB':             '#555555',
    'FullMSE':            '#ff7f0e',
    'FullSPO':            '#17becf',
    'BayesianMSE':        '#d62728',
    'BayesianMSE-TSMask': '#e377c2',
    'BayesianSPO':        '#1f77b4',
    'BayesianSPO-TSMask': '#9467bd',
    'BayesianSPO-DI':     '#2ca02c',
    'BayesianSPO-DFE':    '#ff6600',
    'BayesianSPO-DI+DFE': '#8b0000',
    'BayesianMSE-DI':     '#2ca02c',
    'BayesianMSE-DFE':    '#ff6600',
    'BayesianMSE-DI+DFE': '#8b0000',
}

LINETYPES = {
    'Random': 'dotted', 'Greedy': 'dotted', 'E-Greedy': 'dotted', 'LinUCB': 'dashed',
    'FullMSE': 'dashed', 'FullSPO': 'dashed',
    'BayesianMSE': 'solid', 'BayesianMSE-TSMask': 'dashed',
    'BayesianSPO': 'solid', 'BayesianSPO-TSMask': 'dashed', 'BayesianSPO-DI': 'solid',
    'BayesianSPO-DFE': 'dashed', 'BayesianSPO-DI+DFE': 'solid','BayesianMSE-DI': 'solid',
    'BayesianMSE-DFE': 'dashed', 'BayesianMSE-DI+DFE': 'solid',
}

EXCLUDE_FROM_LINES = ['Random', 'Greedy']
WINDOW = 50

LINE_LEGEND_THEME = theme(legend_position='bottom', legend_title=element_blank(),
                          legend_text=element_text(size=9))

# --- Build per-seed data ---
rows_cum, rows_pr, rows_box = [], [], []
for ds_name, ds_cfg in DATASETS.items():
    ds_results = all_results[ds_name]
    for agent_name, seed_results in ds_results.items():
        for seed in SEEDS:
            result = seed_results[seed]
            for t, r in enumerate(result['cumulative_regret'], 1):
                rows_cum.append(dict(dataset=ds_name, agent=agent_name,
                                     seed=seed, round=t, cumulative_regret=r))
            kernel = np.ones(WINDOW) / WINDOW
            smoothed = np.convolve(result['regrets'], kernel, mode='valid')
            for t, r in enumerate(smoothed):
                rows_pr.append(dict(dataset=ds_name, agent=agent_name,
                                    seed=seed, round=t + WINDOW, per_round_regret=r))
            rows_box.append(dict(
                dataset=ds_name, agent=agent_name, seed=seed,
                group=ds_cfg['group'], noise=ds_cfg['noise'],
                final_cumulative_regret=result['final_regret'],
                last_round_regret=np.mean(result['regrets'][-WINDOW:]),
            ))

df_cum = pd.DataFrame(rows_cum)
df_cum_agg = df_cum.groupby(['dataset', 'agent', 'round']).agg(
    mean=('cumulative_regret', 'mean'), std=('cumulative_regret', 'std')
).reset_index()
df_cum_agg['ymin'] = df_cum_agg['mean'] - df_cum_agg['std']
df_cum_agg['ymax'] = df_cum_agg['mean'] + df_cum_agg['std']

df_pr = pd.DataFrame(rows_pr)
df_pr_agg = df_pr.groupby(['dataset', 'agent', 'round']).agg(
    mean=('per_round_regret', 'mean'), std=('per_round_regret', 'std')
).reset_index()
df_pr_agg['ymin'] = df_pr_agg['mean'] - df_pr_agg['std']
df_pr_agg['ymax'] = df_pr_agg['mean'] + df_pr_agg['std']

df_box = pd.DataFrame(rows_box)

LINE_COLORS = {k: v for k, v in COLORS.items() if k not in EXCLUDE_FROM_LINES}
LINE_LINETYPES = {k: v for k, v in LINETYPES.items() if k not in EXCLUDE_FROM_LINES}

W = 1800
ROW_H = 500

all_plots = []
for ds_name in DATASETS:
    d_cum = df_cum_agg[df_cum_agg['dataset'] == ds_name]
    d_pr = df_pr_agg[df_pr_agg['dataset'] == ds_name]
    d_box = df_box[df_box['dataset'] == ds_name].copy()
    noise = DATASETS[ds_name]['noise']
    group = DATASETS[ds_name]['group']
    label = f'{group} (noise={noise})'

    order = (d_box.groupby('agent')['final_cumulative_regret']
             .median().sort_values().index.tolist())
    d_box['agent'] = pd.Categorical(d_box['agent'], categories=order, ordered=True)

    d_cum_filt = d_cum[~d_cum['agent'].isin(EXCLUDE_FROM_LINES)]
    d_pr_filt = d_pr[~d_pr['agent'].isin(EXCLUDE_FROM_LINES)]

    p_cum = (ggplot(d_cum_filt, aes(x='round', y='mean', color='agent', fill='agent'))
        + geom_ribbon(aes(ymin='ymin', ymax='ymax'), alpha=0.15, color='rgba(0,0,0,0)')
        + geom_line(aes(linetype='agent'), size=1.0)
        + scale_color_manual(values=LINE_COLORS) + scale_fill_manual(values=LINE_COLORS)
        + scale_linetype_manual(values=LINE_LINETYPES)
        + guides(color=guide_legend(nrow=2), fill='none', linetype=guide_legend(nrow=2))
        + labs(title=f'{label}: Cumulative Regret', x='Round', y='Cumulative Regret')
        + theme_minimal() + LINE_LEGEND_THEME)

    p_pr = (ggplot(d_pr_filt, aes(x='round', y='mean', color='agent', fill='agent'))
        + geom_ribbon(aes(ymin='ymin', ymax='ymax'), alpha=0.15, color='rgba(0,0,0,0)')
        + geom_line(aes(linetype='agent'), size=1.0)
        + scale_color_manual(values=LINE_COLORS) + scale_fill_manual(values=LINE_COLORS)
        + scale_linetype_manual(values=LINE_LINETYPES)
        + guides(color=guide_legend(nrow=2), fill='none', linetype=guide_legend(nrow=2))
        + labs(title=f'{label}: Per-Round Regret ({WINDOW}-avg)', x='Round', y='Per-Round Regret')
        + theme_minimal() + LINE_LEGEND_THEME)

    p_box_cum = (ggplot(d_box, aes(x='agent', y='final_cumulative_regret', fill='agent'))
        + geom_boxplot(alpha=0.7, width=0.6)
        + geom_jitter(aes(color='agent'), width=0.15, size=2.5, alpha=0.8)
        + scale_fill_manual(values=COLORS) + scale_color_manual(values=COLORS)
        + labs(title=f'{label}: Final Cum. Regret', x='', y='Cumulative Regret')
        + theme_minimal()
        + theme(legend_position='none', axis_text_x=element_text(angle=45, hjust=1)))

    p_box_pr = (ggplot(d_box, aes(x='agent', y='last_round_regret', fill='agent'))
        + geom_boxplot(alpha=0.7, width=0.6)
        + geom_jitter(aes(color='agent'), width=0.15, size=2.5, alpha=0.8)
        + scale_fill_manual(values=COLORS) + scale_color_manual(values=COLORS)
        + labs(title=f'{label}: Avg Last-{WINDOW} Regret', x='', y='Per-Round Regret')
        + theme_minimal()
        + theme(legend_position='none', axis_text_x=element_text(angle=45, hjust=1)))

    all_plots.extend([p_cum, p_pr, p_box_cum, p_box_pr])

n_datasets = len(DATASETS)
total_h = ROW_H * n_datasets

fig = gggrid(all_plots, ncol=4, align=True, hspace=30, vspace=40) + ggsize(W, total_h)
fig.show()
ggsave(fig, os.path.join(FIGDIR, 'sp_di_improvement_overview.svg'))
print(f'Saved: {FIGDIR}/sp_di_improvement_overview.svg')

Saved: lets-plot-images/sp_di_improvement_overview.svg


## Summary Statistics

In [9]:
for ds_name, ds_results in all_results.items():
    print(f'\n{ds_name}')
    print(f'{"Agent":<24s} {"Final Regret":>14s} {"% vs BayesianMSE":>16s}')
    print('-' * 58)

    mse_final = np.mean([ds_results['BayesianMSE'][s]['final_regret'] for s in SEEDS])

    for agent_name in AGENT_ORDER:
        if agent_name not in ds_results:
            continue
        seed_results = ds_results[agent_name]
        final = np.array([seed_results[s]['final_regret'] for s in SEEDS])
        pct = (final.mean() - mse_final) / mse_final * 100
        sign = '+' if pct >= 0 else ''
        print(f'  {agent_name:<22s} {final.mean():>8.1f} +/- {final.std():>5.1f}'
              f'  {sign}{pct:.1f}%')


sp_nw025
Agent                      Final Regret % vs BayesianMSE
----------------------------------------------------------
  Greedy                  26983.5 +/- 3441.1  +517.4%
  FullMSE                 19614.9 +/- 2222.7  +348.8%
  FullSPO                 18702.3 +/- 1343.6  +327.9%
  BayesianMSE              4370.7 +/- 287.2  +0.0%
  BayesianSPO              4021.4 +/- 246.9  -8.0%
  BayesianSPO-DI           4025.5 +/- 291.6  -7.9%
  BayesianSPO-DFE          3816.2 +/- 384.4  -12.7%
  BayesianSPO-DI+DFE       3800.3 +/- 238.7  -13.1%
  BayesianMSE-DI+DFE       4054.5 +/- 340.6  -7.2%


## Flip-Set Monitoring (Structural)

Track diagnostics at two levels:
- **Raw (per-edge):** |D_t|, ρ_t — counts individual edges in symmetric difference
- **Structural (per-segment):** |D_t^struct|, ρ_t^struct — counts connected components (divergent subpath segments)

In SP, a flip set of 18 edges might be just 2 rerouted segments. The structural metric captures the actual number of decision disagreements. Each segment is "observed" if the selected path traverses at least one edge from it.

Theory predicts: structural ρ_t should be higher than raw ρ_t, since observing *any* edge from a divergent segment provides information about the whole segment.

In [10]:
from collections import defaultdict, deque


def edge_to_nodes(edge_idx, n):
    """Map edge index to (node_u, node_v) pair on an n×n grid.

    Right edges: idx = i*(n-1) + j  →  (i,j) -- (i,j+1)
    Down edges:  idx = n*(n-1) + i*n + j  →  (i,j) -- (i+1,j)
    Nodes encoded as i*n + j.
    """
    n_right = n * (n - 1)
    if edge_idx < n_right:
        i = edge_idx // (n - 1)
        j = edge_idx % (n - 1)
        return (i * n + j, i * n + j + 1)
    else:
        idx = edge_idx - n_right
        i = idx // n
        j = idx % n
        return (i * n + j, (i + 1) * n + j)


def structural_flipset(flip_edge_indices, n):
    """Compute connected components of flip set edges on an n×n grid.

    Returns list of components, each a set of edge indices.
    """
    if len(flip_edge_indices) == 0:
        return []

    adj = defaultdict(set)
    for e in flip_edge_indices:
        u, v = edge_to_nodes(e, n)
        adj[u].add((v, e))
        adj[v].add((u, e))

    visited_nodes = set()
    components = []

    for start_node in adj:
        if start_node in visited_nodes:
            continue
        comp_edges = set()
        queue = deque([start_node])
        visited_nodes.add(start_node)
        while queue:
            node = queue.popleft()
            for neighbor, edge in adj[node]:
                comp_edges.add(edge)
                if neighbor not in visited_nodes:
                    visited_nodes.add(neighbor)
                    queue.append(neighbor)
        components.append(comp_edges)

    return components


def run_experiment_monitored(env, agent, n_rounds, lr, solver, grid_size,
                             is_maximize=False):
    """Run experiment with raw + structural flip-set monitoring + soft α metrics."""
    # Raw metrics
    flipset_sizes = []
    flipset_rhos = []
    pred_errors_inf = []
    # Structural metrics
    struct_sizes = []
    struct_rhos = []
    # α metrics
    alpha_exact = []   # 1{w*(Ỹ) == w*(c)} — exact match (binary)
    alpha_soft = []    # |w*(Ỹ) ∩ w*(c)| / k — edge overlap fraction
    alpha_struct = []  # fraction of divergent segments where imputed agrees with true

    regrets = []
    cumulative_regret = []
    total_regret = 0.0
    X = env.reset()

    for t in range(n_rounds):
        c_pred = agent.predict_mean(X)
        selection = agent.act(X)
        observed_values, reward, info = env.step(selection, full_feedback=False)

        # True costs
        if 'true_costs' in info:
            c_true = info['true_costs'].flatten()
        elif 'true_values' in info:
            c_true = info['true_values'].flatten()
        else:
            c_true = observed_values

        c_pred_np = np.asarray(c_pred, dtype=np.float64).flatten()
        c_true_np = np.asarray(c_true, dtype=np.float64).flatten()
        sel_np = np.asarray(selection).flatten()

        w_true = solver(c_true_np)
        w_spo = solver(2 * c_pred_np - c_true_np)

        # --- Raw flip set (per-edge) ---
        D = np.abs(w_spo - w_true) > 0.5
        D_size = int(D.sum())
        S = sel_np > 0.5

        if D_size > 0:
            rho = float(np.sum(D & S)) / D_size
        else:
            rho = 1.0

        if S.any():
            err_inf = float(np.max(np.abs(c_pred_np[S] - c_true_np[S])))
        else:
            err_inf = 0.0

        flipset_sizes.append(D_size)
        flipset_rhos.append(rho)
        pred_errors_inf.append(err_inf)

        # --- Structural flip set (connected components) ---
        flip_edges = np.where(D)[0]
        obs_edges = set(np.where(S)[0])
        components = structural_flipset(flip_edges, grid_size)
        n_comp = len(components)

        if n_comp > 0:
            n_obs_comp = sum(1 for comp in components if comp & obs_edges)
            struct_rho = n_obs_comp / n_comp
        else:
            struct_rho = 1.0

        struct_sizes.append(n_comp)
        struct_rhos.append(struct_rho)

        # --- α metrics: imputed target quality ---
        Y_imputed = np.where(S, c_true_np, c_pred_np)
        w_imputed = solver(Y_imputed)

        w_true_bin = (w_true > 0.5)
        w_imp_bin = (w_imputed > 0.5)

        # α_exact: binary — does imputed path exactly match true optimal?
        alpha_ex = int(np.array_equal(w_true_bin, w_imp_bin))
        alpha_exact.append(alpha_ex)

        # α_soft: |shared edges| / |path length|
        k = max(int(w_true_bin.sum()), 1)
        shared = int((w_true_bin & w_imp_bin).sum())
        alpha_soft.append(shared / k)

        # α_struct: fraction of divergent segments where imputed agrees with true
        # Divergent segments = connected components of symmetric difference
        # between w_imputed and w_true
        imp_vs_true_diff = w_true_bin ^ w_imp_bin
        diff_edges = np.where(imp_vs_true_diff)[0]
        diff_components = structural_flipset(diff_edges, grid_size)
        n_diff_comp = len(diff_components)

        if n_diff_comp == 0:
            # Perfect agreement — all segments correct
            alpha_struct.append(1.0)
        else:
            # How many of the SPO flip set segments does the imputed solution
            # get right? Compare: for each segment in the SPO flip set,
            # does the imputed solution agree with the true solution on that segment?
            if n_comp > 0:
                n_correct = 0
                for comp in components:
                    # Check if imputed == true on all edges in this segment
                    all_agree = all(w_true_bin[e] == w_imp_bin[e] for e in comp)
                    if all_agree:
                        n_correct += 1
                alpha_struct.append(n_correct / n_comp)
            else:
                alpha_struct.append(1.0)

        # --- Standard tracking ---
        instant_regret = info['regret']
        total_regret += instant_regret
        regrets.append(instant_regret)
        cumulative_regret.append(total_regret)

        try:
            agent.update(X, selection, observed_values, lr, full_feedback=False)
        except TypeError:
            agent.update(X, selection, observed_values, full_feedback=False)

        X = env.get_context()

    return {
        'regrets': regrets,
        'cumulative_regret': cumulative_regret,
        'final_regret': total_regret,
        'flipset_sizes': flipset_sizes,
        'flipset_rhos': flipset_rhos,
        'pred_errors_inf': pred_errors_inf,
        'struct_sizes': struct_sizes,
        'struct_rhos': struct_rhos,
        'alpha_exact': alpha_exact,
        'alpha_soft': alpha_soft,
        'alpha_struct': alpha_struct,
    }


# Run monitored experiments
MONITOR_AGENTS = ['BayesianMSE', 'BayesianSPO', 'BayesianSPO-DI',
                  'BayesianSPO-DFE', 'BayesianSPO-DI+DFE']
MONITOR_SEEDS = [0, 1, 2]

monitor_results = {}
for ds_name, ds_cfg in DATASETS.items():
    env_cls = ds_cfg['env_cls']
    env_kwargs = ds_cfg['env_kwargs']
    minimize = ds_cfg['minimize']
    solver_fn = ShortestPathSolver(GRID_SIZE).solve

    print(f'\n=== Flip-Set Monitoring: {ds_name} ===')
    monitor_results[ds_name] = {}

    for seed in MONITOR_SEEDS:
        agents = make_agents(env_cls(**env_kwargs, seed=seed), seed, minimize=minimize)

        for agent_name in MONITOR_AGENTS:
            if agent_name not in agents:
                continue
            if agent_name not in monitor_results[ds_name]:
                monitor_results[ds_name][agent_name] = {}

            env_run = env_cls(**env_kwargs, seed=seed)
            result = run_experiment_monitored(
                env_run, agents[agent_name], N_ROUNDS, HP['lr'],
                solver=solver_fn, grid_size=GRID_SIZE,
                is_maximize=not minimize)
            monitor_results[ds_name][agent_name][seed] = result
            avg_D = np.mean(result['flipset_sizes'])
            avg_Ds = np.mean(result['struct_sizes'])
            avg_a_soft = np.mean(result['alpha_soft'])
            avg_a_struct = np.mean(result['alpha_struct'])
            print(f'  {agent_name:<22s} seed={seed}'
                  f'  |D|={avg_D:.1f} ({avg_Ds:.1f} seg)'
                  f'  α_soft={avg_a_soft:.3f}'
                  f'  α_struct={avg_a_struct:.3f}')


=== Flip-Set Monitoring: sp_nw025 ===
  BayesianMSE            seed=0  |D|=21.7 (1.3 seg)  α_soft=0.678  α_struct=0.470
  BayesianSPO            seed=0  |D|=20.5 (1.2 seg)  α_soft=0.676  α_struct=0.468
  BayesianSPO-DI         seed=0  |D|=21.2 (1.3 seg)  α_soft=0.678  α_struct=0.473
  BayesianSPO-DFE        seed=0  |D|=23.2 (1.2 seg)  α_soft=0.682  α_struct=0.488
  BayesianSPO-DI+DFE     seed=0  |D|=23.2 (1.2 seg)  α_soft=0.673  α_struct=0.465
  BayesianMSE            seed=1  |D|=19.6 (1.3 seg)  α_soft=0.675  α_struct=0.396
  BayesianSPO            seed=1  |D|=17.9 (1.4 seg)  α_soft=0.678  α_struct=0.417
  BayesianSPO-DI         seed=1  |D|=16.4 (1.1 seg)  α_soft=0.664  α_struct=0.366
  BayesianSPO-DFE        seed=1  |D|=23.7 (1.1 seg)  α_soft=0.697  α_struct=0.394
  BayesianSPO-DI+DFE     seed=1  |D|=23.6 (1.1 seg)  α_soft=0.691  α_struct=0.397
  BayesianMSE            seed=2  |D|=22.1 (1.3 seg)  α_soft=0.752  α_struct=0.548
  BayesianSPO            seed=2  |D|=13.2 (0.9 seg)  α_soft

In [11]:
# --- Flip-Set Monitoring Plots (Raw + Structural + Soft α) ---
MONITOR_WINDOW = 50

MONITOR_COLORS = {
    'BayesianMSE':        '#d62728',
    'BayesianSPO':        '#1f77b4',
    'BayesianSPO-DI':     '#2ca02c',
    'BayesianSPO-DFE':    '#ff6600',
    'BayesianSPO-DI+DFE': '#8b0000',
}

for ds_name in monitor_results:
    rows_fs = []
    for agent_name, seed_results in monitor_results[ds_name].items():
        for seed, result in seed_results.items():
            kernel = np.ones(MONITOR_WINDOW) / MONITOR_WINDOW
            Ds = np.convolve(result['flipset_sizes'], kernel, mode='valid')
            rhos = np.convolve(result['flipset_rhos'], kernel, mode='valid')
            errs = np.convolve(result['pred_errors_inf'], kernel, mode='valid')
            Ds_s = np.convolve(result['struct_sizes'], kernel, mode='valid')
            a_soft = np.convolve(result['alpha_soft'], kernel, mode='valid')
            a_struct = np.convolve(result['alpha_struct'], kernel, mode='valid')
            for t in range(len(Ds)):
                rows_fs.append(dict(
                    agent=agent_name, seed=seed, round=t + MONITOR_WINDOW,
                    flipset_size=Ds[t], rho=rhos[t], pred_error=errs[t],
                    struct_size=Ds_s[t],
                    alpha_soft=a_soft[t], alpha_struct=a_struct[t]))

    df_fs = pd.DataFrame(rows_fs)
    df_fs_agg = df_fs.groupby(['agent', 'round']).agg(
        D_mean=('flipset_size', 'mean'),
        rho_mean=('rho', 'mean'),
        err_mean=('pred_error', 'mean'),
        Ds_mean=('struct_size', 'mean'),
        a_soft_mean=('alpha_soft', 'mean'),
        a_struct_mean=('alpha_struct', 'mean'),
    ).reset_index()

    # Row 1: Raw flip set + prediction error
    p_D = (ggplot(df_fs_agg, aes(x='round', y='D_mean', color='agent'))
        + geom_line(size=1.0)
        + scale_color_manual(values=MONITOR_COLORS)
        + labs(title=f'|D_t| raw (edges)', x='Round', y='|D_t|')
        + theme_minimal() + LINE_LEGEND_THEME)

    p_rho = (ggplot(df_fs_agg, aes(x='round', y='rho_mean', color='agent'))
        + geom_line(size=1.0)
        + geom_hline(yintercept=0.5, linetype='dashed', color='gray', alpha=0.7)
        + scale_color_manual(values=MONITOR_COLORS)
        + labs(title=f'ρ_t raw (per-edge)', x='Round', y='ρ_t')
        + theme_minimal() + LINE_LEGEND_THEME)

    p_err = (ggplot(df_fs_agg, aes(x='round', y='err_mean', color='agent'))
        + geom_line(size=1.0)
        + scale_color_manual(values=MONITOR_COLORS)
        + labs(title=f'‖ĉ−c‖_∞ on observed', x='Round', y='‖ĉ−c‖_∞')
        + theme_minimal() + LINE_LEGEND_THEME)

    # Row 2: Structural size + soft α metrics
    p_Ds = (ggplot(df_fs_agg, aes(x='round', y='Ds_mean', color='agent'))
        + geom_line(size=1.0)
        + scale_color_manual(values=MONITOR_COLORS)
        + labs(title=f'|D_t^struct| (divergent segments)', x='Round', y='# segments')
        + theme_minimal() + LINE_LEGEND_THEME)

    p_a_soft = (ggplot(df_fs_agg, aes(x='round', y='a_soft_mean', color='agent'))
        + geom_line(size=1.0)
        + scale_color_manual(values=MONITOR_COLORS)
        + labs(title='α_soft = |w*(Ỹ) ∩ w*(c)| / k  (edge overlap)',
               x='Round', y=f'α_soft ({MONITOR_WINDOW}-avg)')
        + theme_minimal() + LINE_LEGEND_THEME)

    p_a_struct = (ggplot(df_fs_agg, aes(x='round', y='a_struct_mean', color='agent'))
        + geom_line(size=1.0)
        + scale_color_manual(values=MONITOR_COLORS)
        + labs(title='α_struct (fraction of contested segments correct)',
               x='Round', y=f'α_struct ({MONITOR_WINDOW}-avg)')
        + theme_minimal() + LINE_LEGEND_THEME)

    fig_fs = gggrid([p_D, p_rho, p_err, p_Ds, p_a_soft, p_a_struct],
                    ncol=3, align=True, hspace=30, vspace=30) + ggsize(1800, 900)
    fig_fs.show()
    ggsave(fig_fs, os.path.join(FIGDIR, f'sp_flipset_structural_{ds_name}.svg'))
    print(f'Saved: {FIGDIR}/sp_flipset_structural_{ds_name}.svg')

Saved: lets-plot-images/sp_flipset_structural_sp_nw025.svg
